# Lab 2 - RSA & Secure Messaging

## Aim

- Implement a **hybrid encryption** scheme:
  - RSA for key exchange (public-key cryptography).
  - AES for bulk encryption (symmetric cryptography).
- Send an encrypted message over a TCP socket.

## Overview of Steps

1. Generate an RSA key pair and save it to `private_key.pem` and `public_key.pem`.
2. Start a TCP server that:
   - Receives an encrypted AES key, IV, and ciphertext.
   - Decrypts the AES key with the RSA private key.
   - Decrypts the message with AES and prints it.
3. Run a client that:
   - Loads the public key.
   - Encrypts a plaintext message with AES (CFB mode).
   - Encrypts the AES key with the RSA public key (OAEP padding).
   - Sends everything as a single payload to the server.


In [1]:
# RSA key generation

from cryptography.hazmat.primitives import serialization
from cryptography.hazmat.primitives.asymmetric import rsa

# Generate a private key
private_key = rsa.generate_private_key(public_exponent=65537, key_size=2048)

# Save private key
with open("private_key.pem", "wb") as f:
    f.write(
        private_key.private_bytes(
            encoding=serialization.Encoding.PEM,
            format=serialization.PrivateFormat.PKCS8,
            encryption_algorithm=serialization.NoEncryption(),
        )
    )

# Save public key
public_key = private_key.public_key()
with open("public_key.pem", "wb") as f:
    f.write(
        public_key.public_bytes(
            encoding=serialization.Encoding.PEM,
            format=serialization.PublicFormat.SubjectPublicKeyInfo,
        )
    )

print("Keys saved: private_key.pem, public_key.pem")


Keys saved: private_key.pem, public_key.pem


### Server - receive and decrypt message

The server:

- Listens on `localhost:65432`.
- Accepts one incoming connection.
- Receives a pickled payload containing `(encrypted_key, iv, encrypted_message)`.
- Uses the RSA **private key** to decrypt the AES key.
- Uses AES (CFB mode) with the recovered key and IV to decrypt the ciphertext.
- Prints the plaintext message.

To keep the notebook responsive, we run the server in a **background thread**.


In [2]:
import socket
import pickle
import threading

from cryptography.hazmat.primitives import serialization, hashes
from cryptography.hazmat.primitives.asymmetric import padding
from cryptography.hazmat.primitives.ciphers import Cipher, algorithms, modes

HOST = "localhost"
PORT = 65432

def run_server():
    # Load private key
    with open("private_key.pem", "rb") as f:
        private_key = serialization.load_pem_private_key(f.read(), password=None)

    # Start server
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.bind((HOST, PORT))
        s.listen(1)
        print("Server waiting for connection...")
        conn, addr = s.accept()
        with conn:
            print(f"Connected by {addr}")
            data = b""
            while True:
                chunk = conn.recv(4096)
                if not chunk:
                    break
                data += chunk

    # Unpack payload
    encrypted_key, iv, encrypted_message = pickle.loads(data)

    # Decrypt AES key with RSA private key
    aes_key = private_key.decrypt(
        encrypted_key,
        padding.OAEP(
            mgf=padding.MGF1(algorithm=hashes.SHA256()),
            algorithm=hashes.SHA256(),
            label=None,
        ),
    )

    # Decrypt message with AES (CFB)
    cipher = Cipher(algorithms.AES(aes_key), modes.CFB(iv))
    decryptor = cipher.decryptor()
    message = decryptor.update(encrypted_message) + decryptor.finalize()

    print("Decrypted message:", message.decode(errors="replace"))

# Run server in a background thread
server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()


Server waiting for connection...
Connected by ('127.0.0.1', 58226)
Decrypted message: Hello from the secure sender! This is confidential.


### Client - encrypt and send message

The client:

- Loads the recipient's public key.
- Chooses a plaintext message.
- Generates a random AES key and IV.
- Encrypts the message with AES (CFB mode).
- Encrypts the AES key with the RSA public key using OAEP.
- Packages everything into a tuple and sends it over a TCP socket.

Run this cell **after** the server thread is listening.


In [3]:
import socket
import os
import pickle

from cryptography.hazmat.primitives import serialization, hashes
from cryptography.hazmat.primitives.asymmetric import padding
from cryptography.hazmat.primitives.ciphers import Cipher, algorithms, modes

# Load public key
with open("public_key.pem", "rb") as f:
    public_key = serialization.load_pem_public_key(f.read())

# Message to send
message = b"Hello from the secure sender! This is confidential."

# Generate random AES key and IV
aes_key = os.urandom(32)  # 256-bit AES
iv = os.urandom(16)

# Encrypt message with AES (CFB mode)
cipher = Cipher(algorithms.AES(aes_key), modes.CFB(iv))
encryptor = cipher.encryptor()
encrypted_message = encryptor.update(message) + encryptor.finalize()

# Encrypt AES key with recipient's public key
encrypted_key = public_key.encrypt(
    aes_key,
    padding.OAEP(
        mgf=padding.MGF1(algorithm=hashes.SHA256()),
        algorithm=hashes.SHA256(),
        label=None,
    ),
)

# Package payload
payload = pickle.dumps((encrypted_key, iv, encrypted_message))

# Send via socket
with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
    s.connect((HOST, PORT))
    s.sendall(payload)

print("Encrypted message sent!")


Encrypted message sent!
